# VietOCR FT re-inference — prep + greedy (GPU or CPU)

No retraining. Loads your `vietocr_ft.pth` (vgg_transformer, 37.9M params) and
re-runs OCR with the probe-validated preprocessing:
  - **preprocess**: contrast x1.35 + sharpen, resize to MAX_DIM
  - **greedy batched** decoding (`predict_batch`) — fast, ~beam quality (-0.0009)

**Auto-adapts to your hardware:**
  - **GPU** -> MAX_DIM 1280 (best CER), ~40-60 min for train+test (~6.9k imgs)
  - **CPU** -> MAX_DIM 960 (faster), ~1.5-3 h. Test-only (2,006) is ~half that.

Outputs `ocr_vietocr_ftpb_{test,all}.parquet`. Send BOTH back; the `_all` (train)
file is required to CV-gate against the current 0.6685 OCR before we submit.

## Setup
1. New notebook. **GPU T4 strongly recommended** (Settings > Accelerator). Internet ON.
2. Add Input -> your dataset containing `vietocr_ft.pth`.
3. Run Cell 1 -> **Restart kernel** -> **Save & Run All (Commit)**.
4. Download `ocr_vietocr_ftpb_test.parquet` + `_all` into local `cache/`.

In [ ]:
# Install. Restart kernel after this cell.
!pip install -q easyocr vietocr rapidfuzz
!pip install -q --force-reinstall 'Pillow==10.4.0'
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NOTE: running on CPU -> slower. For speed: Settings > Accelerator > GPU T4.')

In [ ]:
# ===== data discovery + helpers =====
import zipfile, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

KIN = Path('/kaggle/input'); ROOT = None
for p in KIN.rglob('test.csv'):
    ROOT = p.parent; break
assert ROOT is not None, 'competition data not mounted'
WORK = Path('/kaggle/working'); print('ROOT =', ROOT)

def _imgs(kind):
    want_train = (kind == 'train')
    for d in KIN.rglob('*'):
        if d.is_dir() and any(d.glob('img_*.jpg')):
            if ('train' in str(d).lower()) == want_train: return d
    for z in KIN.rglob('*.zip'):
        if ('train' in z.name.lower()) != want_train: continue
        ex = WORK/(kind+'_imgs'); ex.mkdir(parents=True, exist_ok=True)
        if not any(ex.rglob('img_*.jpg')):
            with zipfile.ZipFile(z) as zf: zf.extractall(ex)
        for d in [ex,*[c for c in ex.rglob('*') if c.is_dir()]]:
            if any(d.glob('img_*.jpg')): return d
    raise FileNotFoundError(kind)

TRAIN_DIR = _imgs('train'); TEST_DIR = _imgs('test')
labels = pd.read_csv(ROOT/'train_labels.csv', keep_default_na=False)
test_ids = pd.read_csv(ROOT/'test.csv')['image_id'].tolist()
print('train', len(list(TRAIN_DIR.glob('*.jpg'))), '| test', len(list(TEST_DIR.glob('*.jpg'))))

def clean_ocr(t, maxlen=500):
    t = unicodedata.normalize('NFC', str(t)).replace(chr(10),' ').replace(chr(9),' ')
    t = ' '.join(t.split()); toks = t.split()
    ded = [toks[0]] if toks else []
    for w in toks[1:]:
        if w.lower() != ded[-1].lower(): ded.append(w)
    t = ' '.join(ded)
    return t[:maxlen].rstrip() if len(t) > maxlen else t

In [ ]:
# ===== detector + recognizer (greedy batched) + preprocessing =====
import torch, easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

USE_GPU = torch.cuda.is_available()
DEVICE  = 'cuda:0' if USE_GPU else 'cpu'
MAX_DIM = 1280 if USE_GPU else 960     # GPU: best CER | CPU: faster
print(f'device={DEVICE} | MAX_DIM={MAX_DIM}')

WEIGHTS_PATH = None
for p in KIN.rglob('vietocr_ft.pth'):
    WEIGHTS_PATH = str(p); break
if WEIGHTS_PATH is None:
    for p in KIN.rglob('*.pth'):
        if 'ft' in p.name.lower(): WEIGHTS_PATH = str(p); break
assert WEIGHTS_PATH is not None, 'vietocr_ft.pth not found - attach the weights dataset'
print('weights:', WEIGHTS_PATH)

detector = easyocr.Reader(['vi','en'], gpu=USE_GPU, verbose=False)
def detect_boxes(img):
    try:
        horiz, _ = detector.detect(img, min_size=15, text_threshold=0.6)
        return [tuple(int(v) for v in b) for b in (horiz[0] if horiz else [])]
    except Exception:
        return []

ft_cfg = Cfg.load_config_from_name('vgg_transformer')
ft_cfg['device'] = DEVICE
ft_cfg['weights'] = WEIGHTS_PATH
ft_cfg['cnn']['pretrained'] = False
ft_cfg['predictor']['beamsearch'] = False   # greedy -> predict_batch can batch
ft_rec = Predictor(ft_cfg)
print('loaded ft weights (greedy batched decoding)')

def preprocess(bgr, max_dim=MAX_DIM):
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    w, h = img.size
    if max(w, h) > max_dim:
        r = max_dim/max(w, h); img = img.resize((int(w*r), int(h*r)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35).filter(ImageFilter.SHARPEN)
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

In [ ]:
# ===== ocr_image: preprocess -> detect -> GREEDY BATCH recognize -> order =====
EMPTY = {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}

def ocr_image(path):
    bgr = cv2.imread(str(path))
    if bgr is None: return dict(EMPTY)
    bgr = preprocess(bgr)
    items = []
    for (x0,x1,y0,y1) in detect_boxes(bgr):
        x0,y0 = max(0,x0), max(0,y0); x1,y1 = min(bgr.shape[1],x1), min(bgr.shape[0],y1)
        if x1-x0 < 6 or y1-y0 < 6: continue
        items.append((y0, x0, Image.fromarray(cv2.cvtColor(bgr[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))))
    if not items: return dict(EMPTY)
    texts = ft_rec.predict_batch([it[2] for it in items])   # greedy, batched -> fast
    ys = [it[0] for it in items]; band = max(8.0, (max(ys)-min(ys))/40.0)
    order = sorted(range(len(items)), key=lambda i: (round(items[i][0]/band), items[i][1]))
    text = clean_ocr(' '.join(texts[i] for i in order if str(texts[i]).strip()))
    return {'ocr_text':text, 'raw_text':text, 'mean_conf':0.0, 'n_boxes':len(items), 'n_chars':len(text)}

def run(ids, img_dir, out, save_every=200):
    done = {}
    if Path(out).exists():
        done = {r['image_id']: r for r in pd.read_parquet(out).to_dict('records')}
    recs = list(done.values()); pend = [i for i in ids if i not in done]
    print(out, 'pending', len(pend), '/', len(ids))
    for k, iid in enumerate(tqdm(pend)):
        r = ocr_image(img_dir/(iid+'.jpg')); r['image_id'] = iid; recs.append(r)
        if (k+1) % save_every == 0: pd.DataFrame(recs).to_parquet(out, index=False)
    pd.DataFrame(recs).to_parquet(out, index=False)
    fill = (pd.DataFrame(recs)['ocr_text'].str.strip() != '').mean()
    print('saved', out, '| fill', round(float(fill), 3))

run(test_ids, TEST_DIR, '/kaggle/working/ocr_vietocr_ftpb_test.parquet')
run(labels['image_id'].tolist(), TRAIN_DIR, '/kaggle/working/ocr_vietocr_ftpb_all.parquet')
print('\nDONE - download ocr_vietocr_ftpb_test.parquet (+ _all) into local cache/.')